# LLM Metric Stability

Minimal analysis notebook for saved in-distribution LLM interaction-prediction runs.

This does not rerun LLM inference and does not touch the runner code. It reads saved `predictions.parquet`, treats predictions as fixed, and estimates how stable the reported metrics are under user subsampling/bootstrap.

Main unit of resampling: `user_id`. For the current pointwise interaction setup, the reported main metric is `test.macro_by_user_group_mean.f1`, so the notebook precomputes per-user metric values by averaging candidate-group metrics within each user, then resamples users.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from beyond_click_sim.evaluation import (  # noqa: E402
    binary_classification_metrics,
    grouped_binary_classification_metrics,
    user_grouped_binary_classification_metrics,
)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "in_distribution" / "interaction_prediction"

# Default: current eval_users1000 vLLM full runs. Edit this if you want old full/smoke runs.
RUN_GLOB = "*eval_users1000*llm_yes_no*vllm*full"
SPLIT = "test"

RANDOM_SEED = 20260611
BOOTSTRAP_REPEATS = 1000
SUBSAMPLE_REPEATS = 300
SUBSAMPLE_SIZES = [25, 50, 100, 200, 500, 1000]
METRICS = ["accuracy", "precision", "recall", "f1"]

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## Discover Runs

The table below is intentionally a run-quality check, not just a list. Runs with `llm_errors > 0` or fewer users than the requested eval budget should be interpreted separately.

In [2]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def nested_get(obj: dict, path: list[str], default=None):
    current = obj
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def summarize_run_dir(run_dir: Path) -> dict:
    manifest = read_json(run_dir / "manifest.json")
    metrics = read_json(run_dir / "metrics.json")
    task_manifest = nested_get(manifest, ["task", "manifest"], {}) or {}
    sampler = task_manifest.get("sampler", {})
    splitter = task_manifest.get("splitter", {})
    test_user_metrics = nested_get(metrics, ["test", "macro_by_user_group_mean"], {}) or {}
    test_group_metrics = nested_get(metrics, ["test", "macro_by_group"], {}) or {}
    test_micro_metrics = nested_get(metrics, ["test", "micro"], {}) or {}

    n_users = test_user_metrics.get("n_users")
    requested_users = sampler.get("max_eval_users")
    llm_errors = metrics.get("llm_errors")
    complete_eval = (
        requested_users is not None
        and n_users is not None
        and int(n_users) == int(requested_users)
        and int(llm_errors or 0) == 0
    )

    return {
        "run_id": run_dir.name,
        "run_dir": str(run_dir),
        "dataset": task_manifest.get("dataset"),
        "task": nested_get(manifest, ["task", "name"], metrics.get("task")),
        "method": manifest.get("method", metrics.get("method")),
        "negative_ratio": sampler.get("negative_ratio"),
        "sampler_seed": sampler.get("seed"),
        "splitter_seed": splitter.get("seed"),
        "requested_eval_users": requested_users,
        "n_users": n_users,
        "n_groups": test_user_metrics.get("n_groups", test_group_metrics.get("n_groups")),
        "n_rows": test_user_metrics.get("n", test_micro_metrics.get("n")),
        "scored_rows": metrics.get("scored_rows"),
        "requested_rows": metrics.get("requested_rows"),
        "llm_errors": llm_errors,
        "complete_eval": complete_eval,
        "macro_user_f1": test_user_metrics.get("f1"),
        "macro_user_precision": test_user_metrics.get("precision"),
        "macro_user_recall": test_user_metrics.get("recall"),
        "macro_group_f1": test_group_metrics.get("f1"),
        "micro_f1": test_micro_metrics.get("f1"),
    }


def discover_runs() -> pd.DataFrame:
    run_dirs = sorted(path.parent for path in OUTPUT_ROOT.glob(f"{RUN_GLOB}/predictions.parquet"))
    records = [summarize_run_dir(run_dir) for run_dir in run_dirs]
    if not records:
        raise FileNotFoundError(f"No runs found under {OUTPUT_ROOT} with RUN_GLOB={RUN_GLOB!r}")
    return pd.DataFrame(records).sort_values(
        ["dataset", "negative_ratio", "sampler_seed", "run_id"],
        na_position="last",
    )


runs = discover_runs()
runs[
    [
        "dataset",
        "negative_ratio",
        "sampler_seed",
        "requested_eval_users",
        "n_users",
        "n_groups",
        "n_rows",
        "llm_errors",
        "complete_eval",
        "macro_user_f1",
        "macro_group_f1",
        "micro_f1",
        "run_id",
    ]
]

FileNotFoundError: No runs found under /Users/a.ashabokov/Documents/projects/beyond-click-sim/beyond-click-sim/outputs/in_distribution/interaction_prediction with RUN_GLOB='*eval_users1000*llm_yes_no*vllm*full'

In [ ]:
# Edit this cell to focus on a subset. By default, keep all discovered LLM runs visible,
# including incomplete runs, because incomplete scoring is itself part of run stability.
selected_runs = runs.copy()

# Examples:
# selected_runs = runs.query("complete_eval").copy()
# selected_runs = runs.query("dataset == 'steam'").copy()
# selected_runs = runs.query("dataset == 'ml-1m' and negative_ratio <= 3").copy()

selected_runs[["dataset", "negative_ratio", "n_users", "llm_errors", "complete_eval", "macro_user_f1", "run_id"]]

## Metric Helpers

The scalar metric block below calls the same public metric functions as the runners. The per-user table is the fast representation used for user bootstrap/subsample analysis of `macro_by_user_group_mean` metrics.

In [ ]:
REQUIRED_COLUMNS = {"split", "user_id", "candidate_group", "target", "prediction"}


def load_predictions(run_dir: Path, split: str = SPLIT) -> pd.DataFrame:
    frame = pd.read_parquet(run_dir / "predictions.parquet")
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"{run_dir} predictions.parquet is missing columns: {missing}")
    frame = frame[frame["split"].eq(split)].copy()
    if frame.empty:
        raise ValueError(f"{run_dir} has no rows for split={split!r}")
    frame = frame.dropna(subset=["user_id", "candidate_group", "target", "prediction"])
    frame["target"] = frame["target"].astype(bool)
    frame["prediction"] = frame["prediction"].astype(bool)
    return frame


def compute_metric_blocks(frame: pd.DataFrame) -> dict[str, dict[str, float | int]]:
    y_true = frame["target"]
    y_pred = frame["prediction"]
    return {
        "macro_by_group": grouped_binary_classification_metrics(
            y_true,
            y_pred,
            frame["candidate_group"],
        ),
        "macro_by_user_group_mean": user_grouped_binary_classification_metrics(
            y_true,
            y_pred,
            frame["candidate_group"],
            frame["user_id"],
        ),
        "micro": binary_classification_metrics(y_true, y_pred),
    }


def flatten_metric_blocks(blocks: dict[str, dict[str, float | int]]) -> dict:
    row = {}
    for block_name, values in blocks.items():
        for key, value in values.items():
            row[f"{block_name}.{key}"] = value
    return row


def per_user_metric_values(frame: pd.DataFrame) -> pd.DataFrame:
    work = pd.DataFrame(
        {
            "user_id": frame["user_id"].to_numpy(),
            "candidate_group": frame["candidate_group"].to_numpy(),
            "target": frame["target"].astype(bool).to_numpy(),
            "prediction": frame["prediction"].astype(bool).to_numpy(),
        }
    )
    work["true_positive"] = work["target"] & work["prediction"]

    grouped = work.groupby(["user_id", "candidate_group"], sort=False).agg(
        n=("target", "size"),
        n_positive=("target", "sum"),
        n_predicted_positive=("prediction", "sum"),
        true_positive=("true_positive", "sum"),
    )

    false_positive = grouped["n_predicted_positive"] - grouped["true_positive"]
    true_negative = grouped["n"] - grouped["n_positive"] - false_positive
    grouped["accuracy"] = (grouped["true_positive"] + true_negative) / grouped["n"]
    grouped["precision"] = np.divide(
        grouped["true_positive"],
        grouped["n_predicted_positive"],
        out=np.zeros(len(grouped), dtype=float),
        where=grouped["n_predicted_positive"].to_numpy() != 0,
    )
    grouped["recall"] = np.divide(
        grouped["true_positive"],
        grouped["n_positive"],
        out=np.zeros(len(grouped), dtype=float),
        where=grouped["n_positive"].to_numpy() != 0,
    )
    denom = grouped["precision"] + grouped["recall"]
    grouped["f1"] = np.divide(
        2 * grouped["precision"] * grouped["recall"],
        denom,
        out=np.zeros(len(grouped), dtype=float),
        where=denom.to_numpy() != 0,
    )

    user_metrics = grouped.groupby(level="user_id", sort=False)[METRICS].mean()
    user_metrics["n_groups"] = grouped.groupby(level="user_id", sort=False).size()
    user_metrics["n_rows"] = work.groupby("user_id", sort=False).size()
    return user_metrics.reset_index()

## Load Predictions And Verify Metrics

This checks that the per-user representation reproduces the stored `macro_by_user_group_mean` values up to floating point noise.

In [ ]:
if selected_runs.empty:
    raise ValueError("selected_runs is empty")

scalar_rows = []
user_metric_frames: dict[str, pd.DataFrame] = {}

for run in selected_runs.itertuples(index=False):
    run_dir = Path(run.run_dir)
    frame = load_predictions(run_dir)
    blocks = compute_metric_blocks(frame)
    user_metrics = per_user_metric_values(frame)

    scalar_row = {
        "run_id": run.run_id,
        "dataset": run.dataset,
        "negative_ratio": run.negative_ratio,
        "sampler_seed": run.sampler_seed,
        "n_users_from_predictions": int(user_metrics["user_id"].nunique()),
        **flatten_metric_blocks(blocks),
    }
    for metric in METRICS:
        scalar_row[f"per_user_fast.{metric}"] = float(user_metrics[metric].mean())
        scalar_row[f"stored_macro_user.{metric}"] = getattr(run, f"macro_user_{metric}", np.nan) if metric in {"f1", "precision", "recall"} else np.nan
        scalar_row[f"fast_minus_scalar.{metric}"] = scalar_row[f"per_user_fast.{metric}"] - scalar_row[f"macro_by_user_group_mean.{metric}"]

    scalar_rows.append(scalar_row)
    user_metrics = user_metrics.assign(
        run_id=run.run_id,
        dataset=run.dataset,
        negative_ratio=run.negative_ratio,
        sampler_seed=run.sampler_seed,
    )
    user_metric_frames[run.run_id] = user_metrics

scalar_metrics = pd.DataFrame(scalar_rows).sort_values(["dataset", "negative_ratio", "sampler_seed", "run_id"])
all_user_metrics = pd.concat(user_metric_frames.values(), ignore_index=True)

scalar_metrics[
    [
        "dataset",
        "negative_ratio",
        "sampler_seed",
        "n_users_from_predictions",
        "macro_by_user_group_mean.f1",
        "macro_by_group.f1",
        "micro.f1",
        "fast_minus_scalar.f1",
        "run_id",
    ]
]

## User Bootstrap And Subsampling

- `bootstrap_users_with_replacement`: sample `n_users` users with replacement. Use this for confidence intervals around the full-eval metric.
- `subsample_users_without_replacement`: sample smaller user sets without replacement. Use this for stability curves: how quickly the metric stabilizes as the eval budget grows.

In [ ]:
def sample_means(values: np.ndarray, *, size: int, repeats: int, rng: np.random.Generator, replace: bool) -> np.ndarray:
    n = len(values)
    if not 1 <= size <= n:
        raise ValueError(f"size must be in [1, {n}], got {size}")
    if replace:
        indices = rng.integers(0, n, size=(repeats, size))
        return values[indices].mean(axis=1)
    return np.array(
        [values[rng.choice(n, size=size, replace=False)].mean() for _ in range(repeats)],
        dtype=float,
    )


def summarize_distribution(values: np.ndarray) -> dict[str, float]:
    p025, p50, p975 = np.quantile(values, [0.025, 0.5, 0.975])
    return {
        "mean": float(values.mean()),
        "std": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
        "p025": float(p025),
        "p50": float(p50),
        "p975": float(p975),
        "ci_width_95": float(p975 - p025),
    }


def stability_for_run(user_metrics: pd.DataFrame, *, rng: np.random.Generator) -> pd.DataFrame:
    run_id = str(user_metrics["run_id"].iloc[0])
    dataset = user_metrics["dataset"].iloc[0]
    negative_ratio = user_metrics["negative_ratio"].iloc[0]
    sampler_seed = user_metrics["sampler_seed"].iloc[0]
    n_users = int(len(user_metrics))
    records = []

    subsample_sizes = sorted({size for size in SUBSAMPLE_SIZES if 1 <= size < n_users})
    for metric in METRICS:
        metric_values = user_metrics[metric].to_numpy(dtype=float)
        full_value = float(metric_values.mean())

        boot = sample_means(
            metric_values,
            size=n_users,
            repeats=BOOTSTRAP_REPEATS,
            rng=rng,
            replace=True,
        )
        records.append(
            {
                "run_id": run_id,
                "dataset": dataset,
                "negative_ratio": negative_ratio,
                "sampler_seed": sampler_seed,
                "metric": metric,
                "sample_kind": "bootstrap_users_with_replacement",
                "n_users_total": n_users,
                "n_users_sampled": n_users,
                "repeats": BOOTSTRAP_REPEATS,
                "full_value": full_value,
                **summarize_distribution(boot),
            }
        )

        for size in subsample_sizes:
            means = sample_means(
                metric_values,
                size=size,
                repeats=SUBSAMPLE_REPEATS,
                rng=rng,
                replace=False,
            )
            records.append(
                {
                    "run_id": run_id,
                    "dataset": dataset,
                    "negative_ratio": negative_ratio,
                    "sampler_seed": sampler_seed,
                    "metric": metric,
                    "sample_kind": "subsample_users_without_replacement",
                    "n_users_total": n_users,
                    "n_users_sampled": size,
                    "repeats": SUBSAMPLE_REPEATS,
                    "full_value": full_value,
                    **summarize_distribution(means),
                }
            )

    return pd.DataFrame(records)


rng = np.random.default_rng(RANDOM_SEED)
stability = pd.concat(
    [stability_for_run(frame, rng=rng) for frame in user_metric_frames.values()],
    ignore_index=True,
)

stability.head()

## Full-Eval Bootstrap CI

These rows answer: if this fixed LLM predictor were evaluated on another user sample of the same size, how wide would the metric interval be?

In [ ]:
bootstrap_f1 = stability[
    stability["sample_kind"].eq("bootstrap_users_with_replacement")
    & stability["metric"].eq("f1")
].merge(
    runs[["run_id", "llm_errors", "complete_eval"]],
    on="run_id",
    how="left",
)

bootstrap_f1[
    [
        "dataset",
        "negative_ratio",
        "sampler_seed",
        "n_users_total",
        "llm_errors",
        "complete_eval",
        "full_value",
        "mean",
        "std",
        "p025",
        "p50",
        "p975",
        "ci_width_95",
        "run_id",
    ]
].sort_values(["dataset", "negative_ratio", "sampler_seed", "run_id"])

## Subsample Stability Curve

Use this table to decide how many users are enough for a rough estimate. The key column is `ci_width_95`: smaller means the metric is less sensitive to which users were sampled.

In [ ]:
subsample_f1 = stability[
    stability["sample_kind"].eq("subsample_users_without_replacement")
    & stability["metric"].eq("f1")
].merge(
    runs[["run_id", "llm_errors", "complete_eval"]],
    on="run_id",
    how="left",
)

subsample_f1[
    [
        "dataset",
        "negative_ratio",
        "sampler_seed",
        "n_users_total",
        "n_users_sampled",
        "llm_errors",
        "complete_eval",
        "full_value",
        "mean",
        "std",
        "p025",
        "p975",
        "ci_width_95",
        "run_id",
    ]
].sort_values(["dataset", "negative_ratio", "n_users_sampled", "run_id"])

## All Metrics

`f1` is the main current metric, but precision/recall/accuracy stability can reveal whether a run is stable for the wrong reason, for example because the model predicts positives too often.

In [ ]:
bootstrap_all_metrics = stability[
    stability["sample_kind"].eq("bootstrap_users_with_replacement")
].pivot_table(
    index=["dataset", "negative_ratio", "sampler_seed", "n_users_total", "run_id"],
    columns="metric",
    values=["full_value", "p025", "p975", "ci_width_95"],
)

bootstrap_all_metrics

## User-Level Heterogeneity

This is not a confidence interval. It shows how heterogeneous users are inside each run. High user-level standard deviation means small user subsamples will be noisy even if the full-eval bootstrap CI is acceptable.

In [ ]:
def user_metric_distribution_summary(frame: pd.DataFrame) -> pd.DataFrame:
    records = []
    group_cols = ["dataset", "negative_ratio", "sampler_seed", "run_id"]
    for keys, group in frame.groupby(group_cols, sort=False):
        base = dict(zip(group_cols, keys, strict=True))
        for metric in METRICS:
            values = group[metric].to_numpy(dtype=float)
            records.append(
                {
                    **base,
                    "metric": metric,
                    "n_users": int(len(values)),
                    "mean": float(values.mean()),
                    "std_across_users": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
                    "p05_user": float(np.quantile(values, 0.05)),
                    "p50_user": float(np.quantile(values, 0.5)),
                    "p95_user": float(np.quantile(values, 0.95)),
                }
            )
    return pd.DataFrame(records)


user_distribution = user_metric_distribution_summary(all_user_metrics)
user_distribution[user_distribution["metric"].eq("f1")].sort_values(
    ["dataset", "negative_ratio", "sampler_seed", "run_id"]
)

## Export Tables

Optional: save compact CSVs next to the notebook for reporting or paper tables.

In [ ]:
EXPORT = False

if EXPORT:
    out_dir = PROJECT_ROOT / "outputs" / "analysis" / "llm_metric_stability"
    out_dir.mkdir(parents=True, exist_ok=True)
    runs.to_csv(out_dir / "llm_runs.csv", index=False)
    scalar_metrics.to_csv(out_dir / "scalar_metrics.csv", index=False)
    bootstrap_f1.to_csv(out_dir / "bootstrap_f1.csv", index=False)
    subsample_f1.to_csv(out_dir / "subsample_f1.csv", index=False)
    stability.to_csv(out_dir / "stability_all_metrics.csv", index=False)
    user_distribution.to_csv(out_dir / "user_distribution.csv", index=False)
    print(out_dir)